# SnowJev (Decider): E-Commerce Returns Decision Engine

**Deploy a 4B "System One" decision model on Snowflake using SPCS with GPU**

This notebook deploys [Mapika/decider-4b](https://huggingface.co/Mapika/decider-4b) — an open-source decision model fine-tuned from Qwen3.5-4B-Base — on Snowflake via the Model Registry and SPCS. Once deployed, classify, fraud-check, and route e-commerce return requests directly from SQL.

Unlike text generation, this model reads a state and a set of options in one forward pass and returns calibrated probabilities over the options. There is no decoding, no parsing, and no output outside the options you define.

### Prerequisites

- Run `setup_infra.ipynb` first (creates database, EAIs)
- Connect this notebook to a GPU service on `SYSTEM_COMPUTE_POOL_GPU` (see Section 0)
- ~8.5GB disk space for model weights
- **No Docker required** — Snowflake handles container image building

## 0. Connect to a GPU-backed Service

Before running any cells, connect this notebook to a GPU service:

1. Click **Notebook actions** (top-right) > **Notebook settings**
2. Under **Service settings**, select **GPU**
3. Set **Python version** to `3.12` and **Runtime version** to `v2.10`
4. Set **Compute pool** to `SYSTEM_COMPUTE_POOL_GPU (GPU_NV_S)`
5. Set **Idle timeout** to `1 hour`
6. Under **External access integrations**, enable both `PYPI_ACCESS_INTEGRATION` and `SNOWJEV_EXTERNAL_ACCESS`
7. Click **Create and connect**

Provisioning takes ~2-4 minutes. Once connected, run the cells in order.

## 1. Configuration

In [ ]:
# ---------- Configuration ----------
DATABASE = "SNOWJEV"
SCHEMA = "DECIDER"
COMPUTE_POOL = "SYSTEM_COMPUTE_POOL_GPU"
SERVICE_NAME = "DECIDER_SERVICE"
WAREHOUSE = "COMPUTE_WH"

BASE_MODEL = "Mapika/decider-4b"
BASE_REVISION = "main"

## 2. Connect to Snowflake

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
print(f"Connected as {session.get_current_role()} to {session.get_current_account()}")

## 3. Download Model Weights

`decider` is a proper pip package — no git clone needed. The scoring logic is in the package itself; we only need to download the model weights.

In [ ]:
import os

MODEL_WEIGHTS = os.path.join("/tmp", "_decider_weights")
os.makedirs(MODEL_WEIGHTS, exist_ok=True)

os.environ["HF_HUB_DISABLE_XET"] = "1"
import huggingface_hub.constants
huggingface_hub.constants.HF_HUB_DISABLE_XET = True
from huggingface_hub import snapshot_download
print(f"Downloading {BASE_MODEL} (~8.4GB, cached after first run)...")
snapshot_download(BASE_MODEL, revision=BASE_REVISION, local_dir=MODEL_WEIGHTS)
print(f"Model weights at: {MODEL_WEIGHTS}")

## 4. Define and Log the Custom Model

In [ ]:
%pip install decider-ai torch>=2.4.0 transformers>=5 flash-linear-attention "numpy<2" jinja2

In [ ]:
import json
import pandas as pd
from snowflake.ml.model import custom_model

class DeciderDecisionModel(custom_model.CustomModel):
    def __init__(self, context: custom_model.ModelContext) -> None:
        super().__init__(context)
        from decider.infer import Decider
        model_dir = self.context["model_weights"]
        self.d = Decider(model_dir, device="cuda")

    @custom_model.inference_api
    def decide(self, input_df: pd.DataFrame) -> pd.DataFrame:
        results = []
        for _, row in input_df.iterrows():
            opts_raw = row["OPTIONS"]
            if isinstance(opts_raw, str):
                opts_raw = json.loads(opts_raw)
            options = [o["description"] for o in opts_raw]
            option_ids = [o["id"] for o in opts_raw]
            answer = self.d.decide(
                str(row["STATE"]),
                [{"question": str(row["QUESTION"]), "options": options}]
            )[0]
            probs = [round(answer["probs"].get(opt, 0.0), 4) for opt in options]
            best = max(range(len(probs)), key=probs.__getitem__)
            result = {
                "id": str(row["ID"]),
                "choice": option_ids[best],
                "options": [{"id": oid, "probability": p} for oid, p in zip(option_ids, probs)]
            }
            results.append(json.dumps(result))
        return pd.DataFrame({"DECISION": results})

print("DeciderDecisionModel defined.")

In [ ]:
from snowflake.ml.registry import Registry
from snowflake.ml.model import target_platform
from snowflake.ml.model.model_signature import ModelSignature, FeatureSpec, DataType

model_context = custom_model.ModelContext(model_weights=MODEL_WEIGHTS)
reg = Registry(session=session, database_name=DATABASE, schema_name=SCHEMA)

try:
    m = reg.get_model("DECIDER_DECISION_MODEL")
    if "V1" in [v.version_name for v in m.versions()]:
        m.delete_version("V1")
        print("Deleted existing V1.")
except Exception:
    pass

sample_input = pd.DataFrame([{
    "ID": "test-1", "STATE": "The zipper broke on the first wear.",
    "QUESTION": "Should this return be approved or denied?",
    "OPTIONS": json.dumps([{"id": "approve", "description": "Approve."}, {"id": "deny", "description": "Deny."}]),
}])

print("Logging model with GPU support (~8GB upload, may take 5 min)...\n")
mv = reg.log_model(
    DeciderDecisionModel(model_context),
    model_name="DECIDER_DECISION_MODEL", version_name="V1",
    target_platforms=target_platform.SNOWPARK_CONTAINER_SERVICES_ONLY,
    pip_requirements=[
        "decider-ai",
        "torch>=2.4.0",
        "transformers>=5",
        "flash-linear-attention",
        "numpy<2",
        "huggingface_hub",
        "jinja2",
    ],
    sample_input_data=sample_input,
    options={"cuda_version": "12.3"},
    comment="Decider-4B: System One decision model (one-pass typed decisions with calibrated probabilities)",
)
print(f"Model logged: {DATABASE}.{SCHEMA}.DECIDER_DECISION_MODEL V1")

## 5. Deploy the Model Service on GPU

In [ ]:
print("Deploying model service on GPU (8-10 min)...\n")
try:
    session.sql(f"DROP SERVICE IF EXISTS {DATABASE}.{SCHEMA}.{SERVICE_NAME}").collect()
    print(f"Dropped existing service {SERVICE_NAME} (if any).")
except Exception as e:
    print(f"Note: {e}")

mv.create_service(
    service_name=SERVICE_NAME, service_compute_pool=COMPUTE_POOL,
    gpu_requests="1", ingress_enabled=False,
    build_external_access_integrations=["PYPI_ACCESS_INTEGRATION", "SNOWJEV_EXTERNAL_ACCESS"],
)
print(f"Service {SERVICE_NAME} deployment initiated.")

In [ ]:
import time
timeout, poll_interval, elapsed = 1800, 30, 0
while elapsed < timeout:
    rows = session.sql(f"SELECT SYSTEM$GET_SERVICE_STATUS('{DATABASE}.{SCHEMA}.{SERVICE_NAME}')").collect()
    status_json = json.loads(rows[0][0])
    status = status_json[0].get("status", "UNKNOWN")
    msg = status_json[0].get("message", "")
    print(f"  [{elapsed:>4}s] {status}  {msg}")
    if status == "READY":
        print("\nService is READY.")
        break
    if status in ("FAILED", "DONE"):
        print(f"\nService {status}.")
        break
    time.sleep(poll_interval)
    elapsed += poll_interval
else:
    print("\nTimed out. Check Snowsight > Services & Jobs.")

## 6. Quick Test

In [ ]:
%%sql -r dataframe_1
SELECT SNOWJEV.DECIDER.DECIDER_SERVICE!DECIDE(
    'test-1',
    'I bought this jacket two weeks ago and the zipper broke on the first wear. Stitching is coming apart at the shoulder.',
    'Should this return be approved, denied, or escalated?',
    '[{"id": "approve", "description": "Approve for full refund. Product is defective."}, {"id": "deny", "description": "Deny. Normal wear, outside window, or buyer remorse."}, {"id": "escalate", "description": "Escalate to specialist. Ambiguous or high-value."}]'
) AS decision;

## 7. Load Demo Data

In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE TABLE SNOWJEV.DECIDER.RETURN_REQUESTS (
    request_id STRING, customer_name STRING, product STRING,
    order_value FLOAT, days_since_purchase INT, customer_message STRING
);

In [ ]:
%%sql -r dataframe_3
INSERT INTO SNOWJEV.DECIDER.RETURN_REQUESTS VALUES
('RET-001', 'Sarah M.', 'Winter Jacket', 129.99, 5, 'The zipper broke on the first wear and the stitching is coming apart at the shoulder seam. Clearly a manufacturing defect.'),
('RET-002', 'James K.', 'Running Shoes', 89.95, 3, 'Received the wrong size. I ordered a 10 and got an 8. The box label says 10 but the shoes inside are definitely an 8.'),
('RET-003', 'Maria L.', 'Blender', 64.99, 1, 'The blender arrived with a cracked base and leaks water everywhere when I try to use it. Damaged during shipping.'),
('RET-004', 'Tom R.', 'Wireless Earbuds', 149.00, 12, 'Left earbud stopped working after a week of normal use. Right one still works fine. Tried all troubleshooting steps.'),
('RET-005', 'Kevin P.', 'Dress Shirt', 59.99, 45, 'I just never got around to wearing this shirt. Still has tags on. I know it has been a while but hoping for a refund.'),
('RET-006', 'Lisa H.', 'Perfume', 85.00, 8, 'I don''t like the scent as much as I thought I would. Used about a quarter of the bottle. Can I return the rest?'),
('RET-007', 'Derek N.', 'Hiking Boots', 179.99, 35, 'Wore these on a two-week hiking trip and the soles are worn down. Feel like they should have lasted longer for the price.'),
('RET-008', 'Anna W.', 'Laptop', 1299.00, 14, 'The laptop gets extremely hot under normal use and the fan runs constantly. Not sure if this is a defect or just how it operates.'),
('RET-009', 'Chris B.', 'Designer Handbag', 450.00, 7, 'I think this might be a counterfeit. The logo stitching looks different from my friend''s bag and the leather feels different.'),
('RET-010', 'Rachel F.', 'Smart Watch', 299.99, 20, 'The watch randomly restarts once or twice a day. Done a factory reset and it still happens. Would prefer replacement over refund.'),
('RET-011', 'Alex T.', 'Gaming Console', 499.99, 2, 'Console was dead on arrival. Plugged it in and nothing happens. No lights, no sound, nothing. Need full refund immediately.'),
('RET-012', 'Jordan M.', 'Camera Lens', 899.00, 6, 'This is my fourth return this month. The lens has a scratch on the front element. I am very careful with my equipment.'),
('RET-013', 'Pat S.', 'Noise-Canceling Headphones', 349.99, 1, 'Product said 40-hour battery life but I only get about 38 hours. This is false advertising and I want a full refund plus compensation.'),
('RET-014', 'Emma D.', 'Carry-On Luggage', 189.99, 2, 'Telescoping handle is stuck and won''t extend. I have a flight in three days and desperately need this working or a replacement overnight.'),
('RET-015', 'Mike J.', 'Cotton T-Shirt 3-Pack', 24.99, 10, 'One of the three shirts has a small hole near the hem. Not a big deal but I''d like a partial refund for just that one shirt.'),
('RET-016', 'Nancy O.', 'Wall Art Print', 39.99, 21, 'The colors look noticeably different from the product photos on the website. The blues look more purple in person.'),
('RET-017', 'David L.', 'Electric Toothbrush', 79.99, 4, 'It works great. My dentist told me I should use a different type of brush for my gum condition. Can I return it even though used?'),
('RET-018', 'Sophie R.', 'Yoga Mat', 45.00, 15, 'The mat has a strong chemical smell that hasn''t gone away after two weeks of airing out. Gives me headaches during practice.'),
('RET-019', 'Brian C.', 'Portable Charger', 29.99, 7, 'It charges my phone but only to about 80% then stops. The listing says it should fully charge an iPhone twice. Does not perform as described.'),
('RET-020', 'Olivia K.', 'Silk Pillowcase Set', 69.99, 11, 'After the first wash following care instructions exactly, the pillowcases shrunk significantly and the color faded.');

---

## Demo Scenarios

### Scenario A: Triage All Returns

In [ ]:
%%sql -r dataframe_6
WITH scored AS (
    SELECT request_id, product, order_value,
        SNOWJEV.DECIDER.DECIDER_SERVICE!DECIDE(
            request_id, customer_message,
            'Should this return be approved, denied, or escalated to a specialist?',
            '[{"id": "approve", "description": "Approve for full refund. Defective, wrong item, or mismatch."}, {"id": "deny", "description": "Deny. Normal wear, outside 30-day window, used, or buyer remorse."}, {"id": "escalate", "description": "Escalate. Ambiguous, high-value, or conflicting info."}]'
        ) AS raw,
        LEFT(customer_message, 70) AS preview
    FROM SNOWJEV.DECIDER.RETURN_REQUESTS
)
SELECT request_id, product, order_value,
    PARSE_JSON(raw['DECISION']):choice::STRING AS verdict,
    ROUND(PARSE_JSON(raw['DECISION']):options[0]:probability::FLOAT, 3) AS p_approve,
    ROUND(PARSE_JSON(raw['DECISION']):options[1]:probability::FLOAT, 3) AS p_deny,
    ROUND(PARSE_JSON(raw['DECISION']):options[2]:probability::FLOAT, 3) AS p_escalate,
    preview
FROM scored ORDER BY verdict, p_approve DESC;

### Scenario B: Fraud Detection Sweep

In [ ]:
%%sql -r dataframe_7
WITH scored AS (
    SELECT request_id, customer_name, product, order_value,
        SNOWJEV.DECIDER.DECIDER_SERVICE!DECIDE(
            request_id, customer_message,
            'Does this return request show signs of fraud, abuse, or suspicious behavior?',
            '[{"id": "suspicious", "description": "Signs of fraud: serial returning, implausible claims, demanding immediate refund on high-value items."}, {"id": "legitimate", "description": "Legitimate complaint with no signs of fraud or abuse."}]'
        ) AS raw
    FROM SNOWJEV.DECIDER.RETURN_REQUESTS
)
SELECT request_id, customer_name, product, order_value,
    PARSE_JSON(raw['DECISION']):choice::STRING AS fraud_signal,
    ROUND(PARSE_JSON(raw['DECISION']):options[0]:probability::FLOAT, 3) AS p_fraud,
    ROUND(PARSE_JSON(raw['DECISION']):options[1]:probability::FLOAT, 3) AS p_legit
FROM scored ORDER BY p_fraud DESC;

### Scenario C: Smart Routing

In [ ]:
%%sql -r dataframe_8
WITH scored AS (
    SELECT request_id, product,
        SNOWJEV.DECIDER.DECIDER_SERVICE!DECIDE(
            request_id, customer_message,
            'Which team should handle this return request?',
            '[{"id": "automated", "description": "Handle automatically. Clear defect, wrong item, obvious policy match."}, {"id": "customer_service", "description": "Route to customer service. Needs empathy, explanation, or partial resolution."}, {"id": "specialist", "description": "Route to specialist. Technical assessment, authenticity check, high-value, or fraud."}]'
        ) AS raw
    FROM SNOWJEV.DECIDER.RETURN_REQUESTS
)
SELECT request_id, product,
    PARSE_JSON(raw['DECISION']):choice::STRING AS routed_to,
    ROUND(PARSE_JSON(raw['DECISION']):options[0]:probability::FLOAT, 3) AS p_auto,
    ROUND(PARSE_JSON(raw['DECISION']):options[1]:probability::FLOAT, 3) AS p_cs,
    ROUND(PARSE_JSON(raw['DECISION']):options[2]:probability::FLOAT, 3) AS p_spec
FROM scored ORDER BY routed_to, request_id;

### Scenario D: Multi-Question Analysis

In [ ]:
%%sql -r dataframe_5
WITH target AS (
    SELECT customer_message AS msg FROM SNOWJEV.DECIDER.RETURN_REQUESTS WHERE request_id = 'RET-009'
),
return_decision AS (
    SELECT 'return_decision' AS q, SNOWJEV.DECIDER.DECIDER_SERVICE!DECIDE('v', msg,
        'Should this return be approved, denied, or escalated?',
        '[{"id": "approve", "description": "Approve for refund."}, {"id": "deny", "description": "Deny the return."}, {"id": "escalate", "description": "Escalate to specialist."}]'
        ) AS raw FROM target
),
fraud_check AS (
    SELECT 'fraud_check' AS q, SNOWJEV.DECIDER.DECIDER_SERVICE!DECIDE('f', msg,
        'Does this return show signs of fraud or abuse?',
        '[{"id": "yes", "description": "Signs of fraud."}, {"id": "no", "description": "Legitimate complaint."}]'
        ) AS raw FROM target
),
authenticity_check AS (
    SELECT 'authenticity_check' AS q, SNOWJEV.DECIDER.DECIDER_SERVICE!DECIDE('a', msg,
        'Should this item be inspected for authenticity before processing?',
        '[{"id": "yes", "description": "Yes, verify authenticity."}, {"id": "no", "description": "No, not needed."}]'
        ) AS raw FROM target
),
escalation_risk AS (
    SELECT 'escalation_risk' AS q, SNOWJEV.DECIDER.DECIDER_SERVICE!DECIDE('s', msg,
        'Is this customer likely to leave a negative review if not handled well?',
        '[{"id": "yes", "description": "Likely to escalate publicly."}, {"id": "no", "description": "Reasonable, unlikely to escalate."}]'
        ) AS raw FROM target
)
SELECT q AS question,
    PARSE_JSON(raw['DECISION']):choice::STRING AS answer,
    raw['DECISION'] AS detail
FROM (
    SELECT * FROM return_decision UNION ALL
    SELECT * FROM fraud_check UNION ALL
    SELECT * FROM authenticity_check UNION ALL
    SELECT * FROM escalation_risk
);

### Scenario E: Confidence Analysis

In [ ]:
%%sql -r dataframe_4
WITH scored AS (
    SELECT request_id, product, order_value,
        SNOWJEV.DECIDER.DECIDER_SERVICE!DECIDE(
            request_id, customer_message,
            'Should this return be approved, denied, or escalated?',
            '[{"id": "approve", "description": "Approve for full refund. Defective, wrong item, or mismatch."}, {"id": "deny", "description": "Deny. Normal wear, outside window, used, or buyer remorse."}, {"id": "escalate", "description": "Escalate. Ambiguous, high-value, or conflicting info."}]'
        ) AS raw
    FROM SNOWJEV.DECIDER.RETURN_REQUESTS
),
parsed AS (
    SELECT request_id, product, order_value,
        PARSE_JSON(raw['DECISION']):choice::STRING AS verdict,
        PARSE_JSON(raw['DECISION']):options[0]:probability::FLOAT AS p_approve,
        PARSE_JSON(raw['DECISION']):options[1]:probability::FLOAT AS p_deny,
        PARSE_JSON(raw['DECISION']):options[2]:probability::FLOAT AS p_escalate
    FROM scored
)
SELECT request_id, product, order_value, verdict,
    ROUND(GREATEST(p_approve, p_deny, p_escalate), 3) AS confidence,
    ROUND(p_approve, 3) AS p_approve,
    ROUND(p_deny, 3) AS p_deny,
    ROUND(p_escalate, 3) AS p_escalate,
    CASE WHEN GREATEST(p_approve, p_deny, p_escalate) < 0.7
         THEN 'NEEDS HUMAN REVIEW' ELSE 'AUTO-PROCESSED' END AS action
FROM parsed ORDER BY confidence ASC;

### Scenario F: Decider vs AI_COMPLETE

Head-to-head comparison on the same return requests using two approaches:
- **Decider Service**: Purpose-built 4B decision model with calibrated probabilities
- **AI_COMPLETE (llama3.1-8b)**: General-purpose LLM prompted to decide

Compares decisions, confidence, and latency.

In [ ]:
%%sql -r comparison_results
-- Step 1: Run Decider vs AI_COMPLETE on the same 5 requests
WITH sample_requests AS (
    SELECT * FROM SNOWJEV.DECIDER.RETURN_REQUESTS
    WHERE request_id IN ('RET-001','RET-005','RET-008','RET-009','RET-013')
),

-- Approach 1: Decider Service
decider_results AS (
    SELECT request_id, product, customer_message,
        SNOWJEV.DECIDER.DECIDER_SERVICE!DECIDE(
            request_id, customer_message,
            'Should this return be approved, denied, or escalated?',
            '[{"id": "approve", "description": "Approve for full refund. Defective, wrong item, or mismatch."}, {"id": "deny", "description": "Deny. Normal wear, outside window, used, or buyer remorse."}, {"id": "escalate", "description": "Escalate. Ambiguous, high-value, or conflicting info."}]'
        ) AS raw
    FROM sample_requests
),
decider_parsed AS (
    SELECT request_id, product,
        'Decider-4B' AS approach,
        PARSE_JSON(raw['DECISION']):choice::STRING AS verdict,
        ROUND(GREATEST(
            PARSE_JSON(raw['DECISION']):options[0]:probability::FLOAT,
            PARSE_JSON(raw['DECISION']):options[1]:probability::FLOAT,
            PARSE_JSON(raw['DECISION']):options[2]:probability::FLOAT
        ), 3) AS confidence
    FROM decider_results
),

-- Approach 2: AI_COMPLETE (llama3.1-8b)
complete_results AS (
    SELECT request_id, product,
        'AI_COMPLETE' AS approach,
        AI_COMPLETE(
            'llama3.1-8b',
            'You are a returns analyst. Given a customer message, respond with ONLY a JSON object: {"verdict": "approve|deny|escalate", "confidence": 0.0-1.0}. Rules: approve = defective/wrong item/mismatch; deny = normal wear/outside 30-day window/used/buyer remorse; escalate = ambiguous/high-value/conflicting info.\n\nCustomer message: ' || customer_message
        )::STRING AS raw
    FROM sample_requests
),
complete_parsed AS (
    SELECT request_id, product, approach,
        COALESCE(TRY_PARSE_JSON(REGEXP_SUBSTR(raw, '\\{[^}]+\\}')):verdict::STRING, 'parse_error') AS verdict,
        COALESCE(TRY_PARSE_JSON(REGEXP_SUBSTR(raw, '\\{[^}]+\\}')):confidence::FLOAT, 0) AS confidence
    FROM complete_results
)

-- Combine results
SELECT * FROM decider_parsed
UNION ALL
SELECT request_id, product, approach, verdict, ROUND(confidence, 3) FROM complete_parsed
ORDER BY request_id, approach;

In [ ]:
-- Step 2: Pivot to side-by-side comparison
WITH comparison AS (
    SELECT * FROM {{comparison_results}}
)
SELECT
    REQUEST_ID,
    PRODUCT,
    MAX(CASE WHEN APPROACH = 'Decider-4B'  THEN VERDICT END) AS decider_verdict,
    MAX(CASE WHEN APPROACH = 'Decider-4B'  THEN CONFIDENCE END) AS decider_conf,
    MAX(CASE WHEN APPROACH = 'AI_COMPLETE'  THEN VERDICT END) AS complete_verdict,
    MAX(CASE WHEN APPROACH = 'AI_COMPLETE'  THEN CONFIDENCE END) AS complete_conf,
    CASE WHEN decider_verdict = complete_verdict
         THEN 'Agree' ELSE 'Disagree' END AS consensus
FROM comparison
GROUP BY REQUEST_ID, PRODUCT
ORDER BY REQUEST_ID;

In [ ]:
%%sql -r comparison_summary
-- Step 3: Summary stats
WITH comparison AS (
    SELECT * FROM {{comparison_results}}
)
SELECT
    APPROACH,
    ROUND(AVG(CONFIDENCE), 3) AS avg_confidence,
    ROUND(MIN(CONFIDENCE), 3) AS min_confidence,
    ROUND(MAX(CONFIDENCE), 3) AS max_confidence,
    COUNT(DISTINCT VERDICT) AS unique_verdicts,
    LISTAGG(DISTINCT VERDICT, ', ') WITHIN GROUP (ORDER BY VERDICT) AS verdicts_used
FROM comparison
GROUP BY APPROACH
ORDER BY avg_confidence DESC;

---

## Cleanup

In [ ]:
# To clean up all resources, uncomment and run:
# mv.delete_service(SERVICE_NAME)
# See setup_infra.ipynb cleanup section for full teardown (EAIs, database, etc.)

---

## Summary

| Component | Detail |
|---|---|
| **Model** | [Mapika/decider-4b](https://huggingface.co/Mapika/decider-4b) v2.1 (4.2B params, BF16, ~8.4GB VRAM) |
| **Inference** | One-pass logit readout — no text generation, calibrated probabilities |
| **Package** | [`decider-ai`](https://pypi.org/project/decider-ai/) (Apache 2.0) |
| **Serving** | Snowflake Model Registry + SPCS (`SYSTEM_COMPUTE_POOL_GPU` / GPU_NV_S) |
| **SQL** | `SNOWJEV.DECIDER.DECIDER_SERVICE!DECIDE(id, state, question, options)` |
| **Latency** | ~50-200ms per decision |